# 5. PydanticOutputParser

The parser that turns the model's text into a **validated Pydantic object** — typed fields, real
Python objects, and errors when the data doesn't fit. The most powerful of the prompt-based parsers.

---

## 1. Simple Definition

> **Kid version:** You give the AI an **instruction card** showing the exact form to fill. When it
> replies, this parser reads the answer, puts it into your form (a Pydantic object), and **checks**
> every box is filled with the right kind of thing (a number where a number belongs, etc.).

**Professional definition:** `PydanticOutputParser` (1) generates format instructions from a Pydantic
schema and (2) parses the model's text into a validated instance of that schema, raising
`OutputParserException` if it doesn't conform.

```python
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str = Field(description="the person's full name")
    age: int = Field(description="age in years")

parser = PydanticOutputParser(pydantic_object=Person)
parser.parse('{"name": "Alice", "age": 30}')   # Person(name='Alice', age=30)
```

---

## 2. Why Does It Exist?

**The problem:** You want **typed, validated** data — `age` as a real `int`, required fields enforced —
not a loose dict of strings. Doing that by hand (parse JSON, then construct + validate) is repetitive.

### Before

```python
raw = model.invoke(prompt).content
data = json.loads(raw)          # may crash on noise
person = Person(**data)          # may crash on wrong types — you handle both
```

### After

```python
parser = PydanticOutputParser(pydantic_object=Person)
person = parser.parse(raw)       # parse + validate in one step → Person object
person.age + 1                   # 31  (already an int)
```

It bundles the format instructions and the parse+validate into one reusable object.

---

## 3. Real-Life Analogy

A **strict form-checker at a passport office** 🛂. Every field must be present and correctly typed
(birth date is a real date, age is a number). Anything malformed is rejected. That's the validation
`PydanticOutputParser` adds on top of parsing.

---

## 4. Where It Fits in LangChain Architecture

```
BaseOutputParser
    │
    ▼
PydanticOutputParser        → validated Pydantic object
```

- Uses your Pydantic schema both to **describe** the format (instructions) and to **validate** the
  parse.
- Compared with siblings: returns a **validated object** (vs `JsonOutputParser`'s dict), but **doesn't
  stream** partials. For a plain dict or streaming, use `JsonOutputParser`.
- On modern models, `.with_structured_output()` often replaces this (see comparison below).

---

## 5. Internal Working

```
  ① BUILD TIME
     get_format_instructions()
        → "Return a JSON instance matching this schema: {properties:{name..,age..}, required:[...]}"
        → injected into the prompt

  ② RUN TIME
     model text: '{"name":"Alice","age":30}'
        │
        ▼
     parse():
        extract JSON → json.loads → dict → Person(**dict)  → VALIDATE types
        │
        ▼
     Person(name='Alice', age=30)   (or raise OutputParserException)
```

---

## 6. Attributes / Methods

### `pydantic_object`  *(required)*

**Definition:** The Pydantic model class describing the target shape (see the schema tips below).

**Why it exists:** It's the schema used for both instructions and validation.

**When developers use it:** Always (the one required argument).

```python
PydanticOutputParser(pydantic_object=Person)
```

---

### `get_format_instructions()`

**Definition:** Returns text describing the required JSON (derived from the schema).

**Why it exists:** The prompt must carry the format so the reply is parseable.

```python
print(parser.get_format_instructions())
```

---

### `parse()`

**Definition:** Converts a raw string into a validated object.

```python
parser.parse('{"name":"Alice","age":30}')
```

---

### `parse_with_prompt()` *(advanced)*

**Definition:** Parse using the original prompt as context (used by retry parsers).

---

## Schema tips that boost accuracy

- **Always add `Field(description=...)`** — these are instructions sent to the model.
- Use **`Optional[...]`/defaults** for fields that may be missing (model returns `None`, not a guess).
- Use **`Literal`/`Enum`** for fixed choices; **validation constraints** (`ge`, `le`, `min_length`).
- A **class docstring** becomes the schema's overall description.

```python
from typing import Optional, List
from pydantic import BaseModel, Field

class Person(BaseModel):
    """Information about a person."""
    name: str = Field(description="the person's full name")
    age: int = Field(description="age in years", ge=0, le=150)
    hobbies: List[str] = Field(default_factory=list, description="list of hobbies")
    email: Optional[str] = Field(default=None, description="email if mentioned")
```

---

## Putting it together

```python
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str = Field(description="full name")
    age: int = Field(description="age in years")
    hobbies: list[str] = Field(description="hobbies")

parser = PydanticOutputParser(pydantic_object=Person)

prompt = PromptTemplate(
    template="Extract info.\n{format_instructions}\nText: {text}\n",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | parser
person = chain.invoke({"text": "Alice is 30 and loves painting and hiking."})
print(person)   # Person(name='Alice', age=30, hobbies=['painting', 'hiking'])
```

---

## `PydanticOutputParser` vs `.with_structured_output()`

| | `PydanticOutputParser` | `model.with_structured_output(Schema)` |
|---|---|---|
| Mechanism | prompt instructions + parse text | native function calling / JSON mode |
| Works on any model | ✅ | ❌ only supported models |
| Reliability | good (model can disobey) | very high (provider-enforced) |
| Extra prompt tokens | yes | minimal |
| Use when | local/older models, full control | modern models — simplest + most reliable |

---


In [2]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

# Initialize the language model
llm = ChatOllama(model="qwen3:8b")

class Person(BaseModel):
    name: str = Field(description='Name of the person')
    age: int = Field(gt=18, description='Age of the person')
    city: str = Field(description='Name of the city the person belongs to')

parser = PydanticOutputParser(pydantic_object=Person)

fictional_name_age_city_template = PromptTemplate(
                                                  template='Generate the name, age and city of a fictional {place} person \n {format_instruction}',
                                                  input_variables=['place'],
                                                  partial_variables={'format_instruction':parser.get_format_instructions()}
                                                 )
print(fictional_name_age_city_template.format(place='sri lankan'))

chain = fictional_name_age_city_template | llm | parser

llm_result = chain.invoke({'place':'sri lankan'})

print(llm_result)

Generate the name, age and city of a fictional sri lankan person 
 The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "Name of the person", "title": "Name", "type": "string"}, "age": {"description": "Age of the person", "exclusiveMinimum": 18, "title": "Age", "type": "integer"}, "city": {"description": "Name of the city the person belongs to", "title": "City", "type": "string"}}, "required": ["name", "age", "city"]}
```
name='Anusha Rajapakse' age=28 city='Colombo'
